In [1]:
import os
import sys
import re
import time

project_root = "/root/work/tenset"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")
    

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build:" + os.environ.get("LD_LIBRARY_PATH", "")

In [ ]:
import numpy as np
sys.path.append("/root/work/tenset/scripts")
from print_programs import return_all_states
from make_dataset import load_and_register_tasks
from tvm import auto_scheduler
from tvm.auto_scheduler.dataset import Dataset, make_dataset_from_log_file
from glob import glob
from unfolding.util_modules.util_manager import seed_everything

# json_file = "/root/work/tenset/dataset/measure_records_tenset/k80/([0bcb8746286db050cd088f375c85372d,1,64,64,128,6,6,32,128,1,64,64,32],cuda).json"
json_file = "/root/work/tenset/dataset/measure_records_tenset/k80/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json"
# json_file = "/root/work/tenset/dataset/measure_records_tenset/k80/([3eb184d18885126bd13d564ef260c820,4,16,16,256,6,6,256,256,1,1,1,256,4,16,16,256,4,16,16,256],cuda).json"
# json_file = "/root/work/tenset/dataset/measure_records_tenset/k80/([8c674f26f66543069d1e1c56cda249f9,4,60,60,256,1,1,256,512,1,1,1,512,4,30,30,512],cuda).json"

# json_file = glob("/root/work/tenset/dataset/measure_records_tenset/k80/([0c9a5ba46ffc5e1a9e5641018527117f,4*.json")[0]
save_dir_name = os.path.basename(json_file).replace(".json", "").replace("[","").replace("]","")
load_and_register_tasks()
print("Loading dataset from", json_file)

Loading dataset from /root/work/tenset/dataset/measure_records_tenset/k80/([0c9a5ba46ffc5e1a9e5641018527117f,4,7,7,160,1,1,160,960,1,1,1,960,4,7,7,960],cuda).json


In [3]:
from unfolding.util_modules.extent.records import state_cost_to_records
states, costs = return_all_states(json_file)
records = state_cost_to_records(states, costs)
print(records.keys())

발견된 공통 (0,1) for문 변수: {'yy_c.2', 'yy_c.1', 'ff_c.0', 'xx_c.2', 'ff_c.1', 'ry.0', 'rx.2', 'nn_c.2', 'xx_c.1', 'rx.0', 'nn_c.1', 'ff_c.2', 'ry.1', 'nn_c.0', 'xx_c.0', 'ry.2', 'yy_c.0', 'rx.1'}
dict_keys(['schedules', 'extents', 'costs', 'unroll', 'all', 'cleaned_schedules'])


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class VAE_regression_head(nn.Module):
    def __init__(self, input_dim, feature_dim=None, latent_dim=64, hidden_dim=256):
        """
        input_dim: 2 * D (v_norm + is_zero concat한 차원)
        latent_dim: latent space 차원
        hidden_dim: MLP hidden 크기
        """
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            
            # 출력은 연속값이니까 activation 없이 그대로
        )

        self.cost_predictor = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def encode(self, x, use_mean=False):

        h = self.encoder(x)
        mean = self.fc_mu(h)
        if not use_mean:
            logvar = self.fc_logvar(h)
        else:
            return mean
        
        return mean, logvar


    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std


    def decode(self, z):
        return self.decoder(z)

    def predict_cost(self, z):
        return self.cost_predictor(z)

    def forward(self, x, use_mean=True):
        mu, logvar = self.encode(x)
        if use_mean:
            z = mu
        else:
            z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        
        cost_pred = self.predict_cost(z)
        return x_recon, mu, logvar, z, cost_pred
    

In [19]:
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from unfolding.search_extent.modules_search.dataset import Extent_Cost_Dataset

train_seed = 2023
seed_everything(train_seed)


input_data = np.log1p(np.array(records["all"], dtype=np.float32))
costs = np.log1p(np.array(records["costs"], dtype=np.float32))

scaler = StandardScaler()
input_data_scaled = scaler.fit_transform(input_data)

X_train, X_val, Y_train, Y_val = train_test_split(
    input_data_scaled, costs, test_size=0.5, random_state=train_seed
)


train_dataset = Extent_Cost_Dataset(X_train, Y_train)
val_dataset   = Extent_Cost_Dataset(X_val, Y_val)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=False)
val_loader   = DataLoader(val_dataset,   batch_size=512, shuffle=False)


In [ ]:
from sklearn.metrics import r2_score
import itertools
import torch
from unfolding.util_modules.loss import reconstruction_loss, kld_loss, infonce_loss, reg_loss
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



input_dim = X_train.shape[-1]
latent_dim = 64
hidden_dim = 256


hyperparameter = {
    'lambda_recon': [1.0],
    'lambda_kld': [0.01],
    'lambda_reg': [0.1],
    'lambda_infonce': [0.01],
    'latent_dim': [64],
    'lr': [1e-3],
}

cnt = 0
epochs = 1000


for hyper in itertools.product(*hyperparameter.values()):
    hyper_config = dict(zip(hyperparameter.keys(), hyper))
    
    cnt += 1
    print("=============================================")
    print(f"Experiment {cnt}/{len(list(itertools.product(*hyperparameter.values())))}")
    print(hyper_config)

    seed_everything(train_seed)

    vae = VAE_regression_head(input_dim=input_dim, latent_dim=hyper_config['latent_dim'], hidden_dim=hidden_dim).to(device)
    optimizer = torch.optim.Adam(vae.parameters(), lr=hyper_config['lr'])

    # early stopping
    best_val_loss = float('inf')
    patience = 30
    patience_counter = 0

    for epoch in range(1, epochs+1):
        vae.train()
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)  # (N, D)
            cost_true = y_batch.to(device).view(-1)
            
            

            x_recon, mu, logvar, z, cost_pred = vae(x_batch, use_mean=False)

            loss = 0
            loss += hyper_config['lambda_recon'] * reconstruction_loss(x_recon, x_batch)
            loss += hyper_config['lambda_kld'] * kld_loss(mu, logvar)
            loss += hyper_config['lambda_reg'] * reg_loss(cost_pred, cost_true)
            loss += hyper_config['lambda_infonce'] * infonce_loss(z, cost_true, tau=0.1)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        vae.eval()
        cost_preds = []
        cost_trues = []
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            cost_true = y_batch.to(device)

            x_recon, mu, logvar, z, cost_pred = vae(x_batch, use_mean=True)
            val_loss = 0
            val_loss += hyper_config['lambda_recon'] * reconstruction_loss(x_recon, x_batch)
            val_loss += hyper_config['lambda_kld'] * kld_loss(mu, logvar)
            val_loss += hyper_config['lambda_reg'] * reg_loss(cost_pred, cost_true)
            val_loss += hyper_config['lambda_infonce'] * infonce_loss(z, cost_true, tau=0.1)
            cost_trues.append(cost_true.detach().cpu().numpy())
            cost_preds.append(cost_pred.detach().cpu().numpy())

        if val_loss < best_val_loss:
            best_val_loss = val_loss.item()
            patience_counter = 0
        else:
            patience_counter += 1
            
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
        
        if epoch % 100 == 0:
            val_recon_r2 = r2_score(x_batch.detach().cpu().numpy(), x_recon.detach().cpu().numpy())
            val_reg_r2 = r2_score(np.concatenate(cost_trues), np.concatenate(cost_preds))
            print(f"Epoch {epoch}:")
            print(f"train loss={loss.item():.4f}, recon={reconstruction_loss(x_recon, x_batch).item():.4f}, kl={kld_loss(mu, logvar).item():.4f}", end="\t\t")
            print(f"val loss={val_loss.item():.4f}, recon={reconstruction_loss(x_recon, x_batch).item():.4f}, kl={kld_loss(mu, logvar).item():.4f}")
            print(f"Recon R2 : {val_recon_r2:.4f}")
            print(f"Reg R2 : {val_reg_r2:.4f}")

Experiment 1/1
{'lambda_recon': 1.0, 'lambda_kld': 0.01, 'lambda_reg': 0.1, 'lambda_infonce': 0.01, 'latent_dim': 64, 'lr': 0.001}
Epoch 100:
train loss=0.0702, recon=0.0681, kl=1.5396		val loss=0.0930, recon=0.0681, kl=1.5396
Recon R2 : 0.5279
Reg R2 : 0.5622
Epoch 200:
train loss=0.0300, recon=0.0175, kl=1.4881		val loss=0.0384, recon=0.0175, kl=1.4881
Recon R2 : 0.5815
Reg R2 : 0.8664
Epoch 300:
train loss=0.0243, recon=0.0133, kl=1.3244		val loss=0.0322, recon=0.0133, kl=1.3244
Recon R2 : 0.5858
Reg R2 : 0.9046
Epoch 400:
train loss=0.0221, recon=0.0115, kl=1.1881		val loss=0.0285, recon=0.0115, kl=1.1881
Recon R2 : 0.5878
Reg R2 : 0.9445
Early stopping at epoch 491
